# Вероятностные распределения в Python

**Дисциплина:** Введение в анализ больших данных  
**Задание 2**

В этой тетради разберём работу с непрерывными распределениями в Python:
- нормальное распределение;
- равномерное распределение;
- распределение Стьюдента;
- построение гистограмм и кривых плотности.

**Инструменты:**
- `numpy` — генерация и базовые операции;
- `scipy.stats` — теоретические распределения (`rvs`, `pdf`, `cdf`, `ppf`);
- `matplotlib` / `seaborn` — визуализация.

## 0. Импорт библиотек и настройка графиков

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

# Фиксируем зерно — результаты воспроизводимы
np.random.seed(2024)

print("NumPy:", np.__version__)
print("Готово")

### Краткая теория

**Распределение** случайной величины задаёт, с какими вероятностями она принимает значения.

Для непрерывных распределений важны:
- **плотность** $f(x)$ — `pdf`;
- **функция распределения** $F(x) = P(X \le x)$ — `cdf`;
- **квантили** — `ppf` (обратная к cdf).

Объекты `scipy.stats.norm`, `stats.uniform`, `stats.t` поддерживают методы:

| Метод | Назначение |
|-------|------------|
| `.rvs(size=n)` | случайная выборка объёма n |
| `.pdf(x)` | плотность в точке x |
| `.cdf(x)` | функция распределения |
| `.ppf(q)` | квантиль уровня q |
| `.mean()`, `.std()` | теоретические моменты |

Параметры: `loc` (сдвиг), `scale` (масштаб). Для `stats.uniform` `scale` — **длина** отрезка, а не правая граница.

---
## 1. Нормальное распределение

Полностью задаётся средним $\mu$ (`loc`) и стандартным отклонением $\sigma$ (`scale`).

### Пример 1. Стандартное нормальное N(0, 1), n = 10

In [ ]:
# Генерируем выборку из N(0, 1) объёма 10
x1 = stats.norm.rvs(loc=0, scale=1, size=10)

print("Выборка x1     :", np.round(x1, 4))
print("Среднее        :", round(x1.mean(), 4))
print("Ст. отклонение :", round(x1.std(ddof=1), 4))

In [ ]:
# Гистограмма + теоретическая плотность N(0, 1)
fig, ax = plt.subplots()
ax.hist(x1, bins=5, density=True, color="steelblue",
        alpha=0.7, edgecolor="white", label="гистограмма")

xs = np.linspace(-4, 4, 300)
ax.plot(xs, stats.norm.pdf(xs, loc=0, scale=1),
        color="crimson", lw=2, label="плотность N(0, 1)")

ax.set_title("Пример 1. Стандартное нормальное N(0, 1), n=10")
ax.set_xlabel("x")
ax.set_ylabel("плотность")
ax.legend()
plt.tight_layout()
plt.show()

### Пример 2. Два вектора N(1, 3) и N(-1, 2), n = 100

In [ ]:
vec_a = stats.norm.rvs(loc=1, scale=3, size=100)    # N(mean=1, sd=3)
vec_b = stats.norm.rvs(loc=-1, scale=2, size=100)   # N(mean=-1, sd=2)

print("а) N(1, 3)  : mean =", round(vec_a.mean(), 3),
      " sd =", round(vec_a.std(ddof=1), 3))
print("б) N(-1, 2) : mean =", round(vec_b.mean(), 3),
      " sd =", round(vec_b.std(ddof=1), 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, sample, mu, sigma, title in [
    (axes[0], vec_a, 1, 3, "а) N(mean=1, sd=3), n=100"),
    (axes[1], vec_b, -1, 2, "б) N(mean=-1, sd=2), n=100"),
]:
    ax.hist(sample, bins=15, density=True, color="mediumseagreen",
            alpha=0.7, edgecolor="white", label="гистограмма")

    xs = np.linspace(sample.min() - 2, sample.max() + 2, 300)
    ax.plot(xs, stats.norm.pdf(xs, loc=mu, scale=sigma),
            color="crimson", lw=2, label="теоретическая плотность")

    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("плотность")
    ax.legend()

plt.tight_layout()
plt.show()

---
## 2. Равномерное распределение

Плотность постоянна на $[a, b]$ и равна нулю вне него.  
В `stats.uniform`: `loc=a`, `scale=b-a`.

### Пример 3. U[0, 1], n = 20

In [ ]:
u = stats.uniform.rvs(loc=0, scale=1, size=20)

print("Выборка u      :", np.round(u, 4))
print("min            :", round(u.min(), 4))
print("max            :", round(u.max(), 4))
print("Среднее        :", round(u.mean(), 4), " (теория: 0.5)")

In [ ]:
fig, ax = plt.subplots()
ax.hist(u, bins=8, density=True, color="goldenrod",
        alpha=0.7, edgecolor="white", label="гистограмма")

xs = np.linspace(-0.2, 1.2, 300)
ax.plot(xs, stats.uniform.pdf(xs, loc=0, scale=1),
        color="darkred", lw=2, label="плотность U[0, 1]")

ax.set_title("Пример 3. Равномерное U[0, 1], n=20")
ax.set_xlabel("x")
ax.set_ylabel("плотность")
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Распределение Стьюдента (t)

Похоже на нормальное, но с более «тяжёлыми» хвостами.  
Единственный параметр — число степеней свободы `df`.

### Пример 4. df = 1 и df = 5

In [ ]:
# Сетка по x из задания — для кривой плотности
x_grid = np.arange(-5, 5, 0.01)

# Выборки из t-распределения (n=200)
t_df1 = stats.t.rvs(df=1, size=200)
t_df5 = stats.t.rvs(df=5, size=200)

print("t(df=1): mean =", round(t_df1.mean(), 3),
      " sd =", round(t_df1.std(ddof=1), 3))
print("t(df=5): mean =", round(t_df5.mean(), 3),
      " sd =", round(t_df5.std(ddof=1), 3))
print("Теоретические sd: df=1 ->", round(stats.t(df=1).std(), 3),
      "| df=5 ->", round(stats.t(df=5).std(), 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, sample, df, title in [
    (axes[0], t_df1, 1, "t-распределение, df=1, n=200"),
    (axes[1], t_df5, 5, "t-распределение, df=5, n=200"),
]:
    ax.hist(sample, bins=30, density=True, color="mediumpurple",
            alpha=0.7, edgecolor="white", label="гистограмма")

    ax.plot(x_grid, stats.t.pdf(x_grid, df=df),
            color="crimson", lw=2, label=f"плотность t(df={df})")

    ax.set_xlim(-5, 5)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("плотность")
    ax.legend()

plt.tight_layout()
plt.show()

### Дополнительно: сравнение t(df=1), t(df=5) и N(0, 1)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x_grid, stats.t.pdf(x_grid, df=1), lw=2, label="t(df=1)")
plt.plot(x_grid, stats.t.pdf(x_grid, df=5), lw=2, label="t(df=5)")
plt.plot(x_grid, stats.norm.pdf(x_grid, 0, 1), "k--", lw=2,
         alpha=0.7, label="N(0, 1)")

plt.title("Сравнение плотностей: t(df=1), t(df=5) и N(0, 1)")
plt.xlabel("x")
plt.ylabel("плотность")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Шпаргалка

| Задача | Код |
|--------|-----|
| Выборка $N(\mu, \sigma)$ | `stats.norm.rvs(loc=mu, scale=sigma, size=n)` |
| Плотность $N(\mu, \sigma)$ | `stats.norm.pdf(x, loc=mu, scale=sigma)` |
| Выборка $U(a, b)$ | `stats.uniform.rvs(loc=a, scale=b-a, size=n)` |
| Плотность $U(a, b)$ | `stats.uniform.pdf(x, loc=a, scale=b-a)` |
| Выборка $t(df)$ | `stats.t.rvs(df=df, size=n)` |
| Плотность $t(df)$ | `stats.t.pdf(x, df=df)` |
| Гистограмма плотности | `plt.hist(x, bins=..., density=True)` |
| Воспроизводимость | `np.random.seed(число)` |